# Multi-Node HPO via `submit_file()` Pattern

**Architecture: Parallelize TRIALS across nodes (not training within each trial)**

The Snowflake ML team confirmed:
- Multi-node `XGBEstimator` inside a Tuner `train_func` is **not yet supported**
- Tuner + single-node XGBoost per trial **is supported**
- Multi-node distributed `XGBEstimator` at the top level (outside Tuner) **is supported**

This notebook uses `submit_file()` to run a standalone HPO script that:
1. Runs single-node XGBoost within each trial (`XGBClassifier` with `n_jobs=-1`)
2. Parallelizes across **trials** using `max_concurrent_trials=4`
3. The 5-node cluster runs 4 trials simultaneously, each on its own node

```
  ML Job (5 nodes, submit_file)

  hpo_job.py runs on the Ray cluster:
    Node 0 (head): Tuner scheduler + trial runner
    Nodes 1-4:     Each runs 1 concurrent trial (XGBoost single-node)

  6 trials total, 4 concurrent -> completes in ~2 rounds
```

**Why `submit_file()` instead of `@remote`?**

The `@remote` decorator uses cloudpickle to serialize functions. When the function
contains nested closures (like `train_func` inside the job function), deserialization
on the cluster can hang silently with no error or timeout. `submit_file()` sends
a standalone .py script — no serialization, no silent failures.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Session: {session.get_current_role()}, WH: {session.get_current_warehouse()}")

In [ ]:
DB = "RRD_ML_DEMO"
SCHEMA = "DISTRIBUTED_TRAINING"
COMPUTE_POOL = "HPO_POOL"
STAGE = f"{DB}.{SCHEMA}.ML_JOB_STAGE"

session.sql(f"CREATE DATABASE IF NOT EXISTS {DB}").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}.{SCHEMA}").collect()
session.sql(f"USE DATABASE {DB}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
session.sql(f"CREATE STAGE IF NOT EXISTS ML_JOB_STAGE").collect()
print(f"Using {DB}.{SCHEMA}")
print(f"Compute pool: {COMPUTE_POOL}")
print(f"Stage: {STAGE}")

In [ ]:
# Create test data if it doesn't exist
try:
    row_count = session.sql("SELECT COUNT(*) as C FROM MULTINODE_TEST_DATA").collect()[0]["C"]
    print(f"MULTINODE_TEST_DATA already exists ({row_count:,} rows)")
except:
    session.sql("""
    CREATE OR REPLACE TABLE MULTINODE_TEST_DATA AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY SEQ4()) AS ID,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(1)) AS FEAT_0,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(2)) AS FEAT_1,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(3)) AS FEAT_2,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(4)) AS FEAT_3,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(5)) AS FEAT_4,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(6)) AS FEAT_5,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(7)) AS FEAT_6,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(8)) AS FEAT_7,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(9)) AS FEAT_8,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(10)) AS FEAT_9,
        IFF(UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(99)) > 0.5, 1, 0) AS TARGET
    FROM TABLE(GENERATOR(ROWCOUNT => 100000))
    """).collect()
    print("Created MULTINODE_TEST_DATA (100K rows, 10 features)")

## Upload the HPO Script to Stage

The `hpo_job.py` file in this workspace contains the self-contained HPO logic.
We upload it to the internal stage so `submit_file()` can reference it.

In [ ]:
# Upload hpo_job.py to stage
session.file.put(
    "file://hpo_job.py",
    f"@{STAGE}/hpo_scripts/",
    auto_compress=False,
    overwrite=True
)
print(f"Uploaded hpo_job.py to @{STAGE}/hpo_scripts/")

# Verify
files = session.sql(f"LIST @{STAGE}/hpo_scripts/").collect()
for f in files:
    print(f"  {f['name']}  ({f['size']} bytes)")

## Submit the HPO Job

Using `submit_file()` — the script runs directly as a Python process on the head node.
No cloudpickle serialization, no closure issues.

**Key parameters:**
- `target_instances=5`: Provisions 5 nodes (1 head + 4 workers for Ray cluster)
- The script's `XGBScalingConfig(num_workers=3)` uses 3 of those as XGB workers
- Remaining nodes handle Tuner scheduling and Ray overhead

In [ ]:
from snowflake.ml.jobs import submit_file

job = submit_file(
    f"@{STAGE}/hpo_scripts/hpo_job.py",
    compute_pool=COMPUTE_POOL,
    stage_name=STAGE,
    session=session,
    target_instances=4,
)

print(f"Job submitted successfully!")
print(f"  Job ID: {job.id}")
print(f"  Status: {job.status}")
print(f"")
print(f"Monitor with:")
print(f"  job.status        - check current status")
print(f"  job.get_logs()    - view execution logs")
print(f"  job.wait()        - block until completion")

## Monitor Job Progress

In [ ]:
# Check job status and recent logs
print(f"Status: {job.status}")
print(f"")
logs = job.get_logs()
# Show last 3000 chars of logs
print(logs[-3000:] if len(logs) > 3000 else logs)

In [ ]:
# Wait for completion (blocking) and get result
result = job.wait()
print(f"Job completed!")
print(f"Result: {result}")

## Why This Pattern Works

| Issue | `@remote` (broken) | `submit_file()` (working) |
|-------|-------------------|---------------------------|
| Serialization | cloudpickle — fails on nested closures + Tuner imports | None — runs .py directly |
| Silent hangs | Deserialization blocks before user code | Script stdout visible immediately |
| Debugging | No print output visible during hang | All print() shows in logs |
| Closure capture | `_feature_cols` may not survive pickle round-trip | Module-level vars, no pickle needed |

### Key Design Decisions in `hpo_job.py`

1. **Materialize data before Tuner** — Write train/test splits to temp tables so DataConnectors don't need a live warehouse during trial execution
2. **`XGBScalingConfig(num_workers=3)`** — Explicitly limit XGB workers to leave room for the Tuner scheduler on the Ray cluster
3. **`max_concurrent_trials=1`** — Run trials sequentially so XGBEstimator gets full access to the 3 distributed workers per trial
4. **All imports inside `train_func()`** — Each trial gets a fresh import context on the worker

### Production Deployment

Schedule as a recurring job with a Task:

```sql
CREATE OR REPLACE TASK RRD_ML_DEMO.DISTRIBUTED_TRAINING.WEEKLY_HPO
  WAREHOUSE = ML_DEMO_WH
  SCHEDULE = 'USING CRON 0 2 * * 0 America/Chicago'
AS
  -- Task triggers the ML Job via stored procedure
  CALL RRD_ML_DEMO.DISTRIBUTED_TRAINING.RUN_HPO_JOB();
```

In [ ]:
COPY INTO @RRD_ML_DEMO.DISTRIBUTED_TRAINING.HPO_BUG_EVIDENCE/HPO_JOB_58F5D809/logs
FROM RRD_ML_DEMO.DISTRIBUTED_TRAINING.HPO_JOB_58F5D809_LOGS
FILE_FORMAT = (TYPE = 'PARQUET')
OVERWRITE = TRUE;

In [ ]:
LIST @RRD_ML_DEMO.DISTRIBUTED_TRAINING.HPO_BUG_EVIDENCE/HPO_JOB_58F5D809/;

In [ ]:
CREATE FILE FORMAT IF NOT EXISTS RRD_ML_DEMO.DISTRIBUTED_TRAINING.PARQUET_FF TYPE = 'PARQUET';

In [ ]:
%%sql -r dataframe_3
SELECT 
    $1:"INSTANCE_ID"::NUMBER as instance_id, 
    SUBSTR($1:"LOGS"::VARCHAR, 1, 1500) as log_preview
FROM @RRD_ML_DEMO.DISTRIBUTED_TRAINING.HPO_BUG_EVIDENCE/HPO_JOB_58F5D809/
(FILE_FORMAT => 'RRD_ML_DEMO.DISTRIBUTED_TRAINING.PARQUET_FF');

In [ ]:
SELECT instance_id, LENGTH(logs) as log_length 
FROM RRD_ML_DEMO.DISTRIBUTED_TRAINING.HPO_JOB_58F5D809_LOGS 
ORDER BY instance_id;

In [ ]:
COPY INTO @RRD_ML_DEMO.DISTRIBUTED_TRAINING.HPO_BUG_EVIDENCE/HPO_JOB_58F5D809/logs
FROM RRD_ML_DEMO.DISTRIBUTED_TRAINING.HPO_JOB_58F5D809_LOGS
FILE_FORMAT = (TYPE = 'PARQUET')
OVERWRITE = TRUE;